# 4 лаба. Сохранение и загрузка нейронной сети

В лабораторной 3 мы обучили сеть **4 → 8 → 3** различать три вида ирисов.
Теперь научимся сохранять её веса и загружать их в новую сеть.

**Работаем в Google Colab.** Запускайте ячейки сверху вниз.
Порядок работы: **обучаем → сохраняем в JSON → создаём такую же сеть → загружаем веса → проверяем ответ**.

В конце — задание: сделать сервисы классификации и регрессии на FastAPI с HTML-формами.


In [1]:
import json
import numpy as np
from sklearn.datasets import load_iris


## 1. Готовим данные, как в лабораторной 3

Используем все 150 ирисов: 120 для обучения и 30 для проверки.
Для нормализации нужны минимумы `mn` и максимумы `mx`, рассчитанные по обучающим данным.


In [2]:
data = load_iris()
names = [str(name) for name in data.target_names]
X = data.data
labels = data.target                 # номера видов: 0, 1, 2

# Перемешиваем и делим 80 / 20
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
X, labels = X[idx], labels[idx]

Y = np.eye(3)[labels]                # one-hot: строчка единичной матрицы с нужным номером

X_train, X_test = X[:120], X[120:]
Y_train, Y_test = Y[:120], Y[120:]
l_train, l_test = labels[:120], labels[120:]

# Нормализация (min и max считаем ТОЛЬКО по обучающей выборке!)
mn, mx = X_train.min(axis=0), X_train.max(axis=0)
X_train = (X_train - mn) / (mx - mn)
X_test = (X_test - mn) / (mx - mn)

print("Учимся на", len(X_train), "цветках, проверяем на", len(X_test))
print("Цветков каждого вида в обучении:", dict(zip(names, np.bincount(l_train).tolist())))
print("Пример ответа:", names[l_train[0]], "→", Y_train[0])


Учимся на 120 цветках, проверяем на 30
Цветков каждого вида в обучении: {'setosa': 37, 'versicolor': 44, 'virginica': 39}
Пример ответа: versicolor → [0. 1. 0.]


In [3]:
Y_train

## 2. Та же нейронная сеть

Классы `Neuron` и `SimpleNeuralNetwork` берём из лабораторной 3 без изменений.
Четыре признака поступают в скрытый слой из восьми нейронов, затем — в три выходных нейрона.


In [4]:
# Функция активации (сигмоида)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Производная сигмоиды.
# Принимает УЖЕ посчитанный ответ нейрона s = sigmoid(x): производная равна s * (1 - s)
def sigmoid_derivative(s):
    return s * (1 - s)


# Класс нейрона — такой же, как в лабе 1
class Neuron:
    def __init__(self, input_size):
        # Случайные веса от -1 до 1: и положительные, и отрицательные,
        # чтобы нейроны с самого начала «смотрели» на признаки по-разному
        self.weights = np.random.uniform(-1, 1, input_size)
        self.bias = np.random.uniform(-1, 1)

    def feedforward(self, inputs):
        total = np.dot(inputs, self.weights) + self.bias   # взвешенная сумма
        return sigmoid(total)                              # ответ от 0 до 1

# Класс простой нейронной сети с одним скрытым слоем
class SimpleNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # Скрытый слой: hidden_size нейронов, каждый смотрит на все входы
        self.hidden_layer = [Neuron(input_size) for _ in range(hidden_size)]
        # Выходной слой: output_size нейронов, каждый смотрит на все скрытые нейроны
        self.output_layer = [Neuron(hidden_size) for _ in range(output_size)]

    def feedforward(self, inputs):
        # Прямой проход через скрытый слой
        hidden_outputs = np.array([neuron.feedforward(inputs) for neuron in self.hidden_layer])
        # Прямой проход через выходной слой
        outputs = np.array([neuron.feedforward(hidden_outputs) for neuron in self.output_layer])
        return outputs

    def predict(self, inputs):
        # Номер вида, чей выходной нейрон выдал самое большое число
        return int(np.argmax(self.feedforward(inputs)))

    def accuracy(self, X, labels):
        # Доля цветков, для которых сеть назвала правильный вид
        predictions = np.array([self.predict(x) for x in X])
        return np.mean(predictions == labels)

    def train(self, inputs, targets, learning_rate, epochs):
        for epoch in range(epochs):
            total_error = 0
            for i in range(len(inputs)):
                # 1. Прямой проход (запоминаем ответы скрытого слоя — они нужны для обучения)
                hidden_outputs = np.array([neuron.feedforward(inputs[i]) for neuron in self.hidden_layer])
                outputs = np.array([neuron.feedforward(hidden_outputs) for neuron in self.output_layer])

                # 2. Ошибка на выходном слое: правильный ответ минус ответ сети
                errors = targets[i] - outputs
                total_error += np.mean(errors ** 2)

                # 3. Поправка для выходных нейронов: ошибка × производная сигмоиды
                d_outputs = errors * sigmoid_derivative(outputs)

                # 4. Ошибка течёт назад: каждый скрытый нейрон k собирает вину
                #    всех выходных нейронов j пропорционально весу связи между ними.
                #    Считаем ДО изменения весов выходного слоя!
                d_hidden = np.array([
                    sum(d_outputs[j] * self.output_layer[j].weights[k] for j in range(len(outputs)))
                    * sigmoid_derivative(hidden_outputs[k])
                    for k in range(len(self.hidden_layer))
                ])

                # 5. Подправляем веса выходного слоя (вход для них — ответы скрытого слоя)
                for j, neuron in enumerate(self.output_layer):
                    neuron.weights += learning_rate * d_outputs[j] * hidden_outputs
                    neuron.bias += learning_rate * d_outputs[j]

                #    ...и веса скрытого слоя (вход для них — признаки цветка)
                for k, neuron in enumerate(self.hidden_layer):
                    neuron.weights += learning_rate * d_hidden[k] * inputs[i]
                    neuron.bias += learning_rate * d_hidden[k]

            # Логирование процесса обучения
            if epoch % 100 == 0 or epoch == epochs - 1:
                mse = total_error / len(inputs)
                acc = self.accuracy(inputs, targets.argmax(axis=1))
                print(f"эпоха {epoch:4d} | ошибка {mse:.4f} | точность на обучении {acc:.0%}")


## 3. Обучаем

Создаём сеть **4 → 8 → 3** и обучаем её. Сначала проверим качество, затем сохраним параметры.


In [5]:
np.random.seed(1)

net = SimpleNeuralNetwork(input_size=4, hidden_size=8, output_size=3)
net.train(X_train, Y_train, learning_rate=0.5, epochs=500)

print(f"Точность на тесте: {net.accuracy(X_test, l_test):.0%}")


эпоха    0 | ошибка 0.2176 | точность на обучении 62%
эпоха  100 | ошибка 0.0163 | точность на обучении 98%
эпоха  200 | ошибка 0.0149 | точность на обучении 98%
эпоха  300 | ошибка 0.0143 | точность на обучении 98%
эпоха  400 | ошибка 0.0140 | точность на обучении 98%
эпоха  499 | ошибка 0.0137 | точность на обучении 98%
Точность на тесте: 97%


## 4. Сохраняем веса в JSON

Сохраняем **веса и смещения каждого нейрона**. Массивы NumPy превращаем в обычные списки через `.tolist()`.
JSON — текстовый файл: его можно открыть и посмотреть содержимое.

В тот же файл запишем `mn`, `mx` и названия видов. Это нужно, чтобы новый цветок
нормализовался так же, как при обучении, а номер класса превратился в название.


In [6]:
saved = {
    "hidden": [
        {"weights": neuron.weights.tolist(), "bias": float(neuron.bias)}
        for neuron in net.hidden_layer
    ],
    "output": [
        {"weights": neuron.weights.tolist(), "bias": float(neuron.bias)}
        for neuron in net.output_layer
    ],
    "mn": mn.tolist(),
    "mx": mx.tolist(),
    "names": names,
}


In [7]:
saved

In [8]:
with open("iris_weights.json", "w", encoding="utf-8") as file:
    json.dump(saved, file, ensure_ascii=False, indent=2)

print("Веса сохранены в iris_weights.json")

Веса сохранены в iris_weights.json


## 5. Создаём такую же сеть и загружаем веса

Архитектуру воспроизводим одной строкой: `SimpleNeuralNetwork(4, 8, 3)`.
При создании веса случайные — заменим их значениями из файла.
`zip` берёт по очереди нейрон новой сети и сохранённые параметры этого нейрона.

**Важно:** архитектура должна совпадать с обученной сетью. Если обучили, например, 4 → 16 → 3,
при загрузке тоже создайте 4 → 16 → 3. В нашем примере всегда используется **4 → 8 → 3**.
Повторно вызывать `train()` не нужно.


In [9]:
with open("iris_weights.json", "r", encoding="utf-8") as file:
    saved = json.load(file)

new_net = SimpleNeuralNetwork(input_size=4, hidden_size=8, output_size=3)

for neuron, params in zip(new_net.hidden_layer, saved["hidden"]):
    neuron.weights = np.array(params["weights"])
    neuron.bias = params["bias"]

for neuron, params in zip(new_net.output_layer, saved["output"]):
    neuron.weights = np.array(params["weights"])
    neuron.bias = params["bias"]

new_mn = np.array(saved["mn"])
new_mx = np.array(saved["mx"])
new_names = saved["names"]

print("Веса загружены в новую сеть")


Веса загружены в новую сеть


## 6. Проверяем ответ

Передадим один цветок исходной и загруженной сети. Ответы должны совпасть.
Порядок признаков: **длина чашелистика, ширина чашелистика, длина лепестка, ширина лепестка**, всё в сантиметрах.
Измените числа и попробуйте определить другой цветок.


In [10]:
flower = np.array([6.7, 3.3, 5.7, 2.1])
prediction = new_net.feedforward((flower - new_mn) / (new_mx - new_mn))
print("Выходы сигмоидных нейронов (не one-hot):", np.round(prediction, 4))
print("Вид ириса:", new_names[int(np.argmax(prediction))])
before = np.array([net.feedforward(x) for x in X_test])
after = np.array([new_net.feedforward(x) for x in X_test])
np.testing.assert_allclose(before, after, rtol=0, atol=0)
print("Выходы до и после сохранения совпадают на всех 30 тестовых объектах.")


Выходы сигмоидных нейронов (не one-hot): [6.000e-04 9.700e-03 9.943e-01]
Вид ириса: virginica
Выходы до и после сохранения совпадают на всех 30 тестовых объектах.


## 7. Скачиваем файл из Colab

Скачайте `iris_weights.json`: он понадобится для приложения FastAPI.
После завершения сессии Colab файлы в рабочей папке могут исчезнуть.


In [11]:
try:
    from google.colab import files
except ImportError:
    print("Файл iris_weights.json сохранён в текущей папке.")
else:
    files.download("iris_weights.json")


Файл iris_weights.json сохранён в текущей папке.


Если хотите проверить загрузку **после перезапуска Colab**, выполните ячейки с импортами и классами сети
(раздел 2), затем загрузите скачанный файл через панель **Файлы → Загрузить** слева.
После этого выполните раздел 5: сеть восстановится без подготовки обучающих данных и без обучения.
Проверить её можно так:

```python
flower = np.array([6.7, 3.3, 5.7, 2.1])
x = (flower - new_mn) / (new_mx - new_mn)
print(new_names[new_net.predict(x)])
```


## 8. Задание: FastAPI и HTML-формы

**Разработайте сервис классификации объектов и сервис регрессии на FastAPI.
В регрессионной нейронной сети должен быть ровно один выходной нейрон.**
Обе задачи можно разместить в одном приложении с двумя страницами.

### Классификация

1. Обучите нейросеть в Colab. Можно использовать ирисы из этой лабораторной или выбрать, что будете классифицировать.
2. Сохраните веса, смещения и параметры нормализации в JSON. Скачайте файл.
3. В приложении создайте такую же сеть и загрузите параметры из файла при запуске.
4. Сделайте HTML-форму для ввода признаков. После отправки формы показывайте на странице предсказанный класс.

### Регрессия

1. Выберите одну числовую величину для предсказания. Например, длину лепестка ириса
   по трём другим измерениям. Саму длину лепестка не передавайте на вход.
2. Обучите сеть с **одним выходным нейроном и линейной активацией на выходе**.
   Пример архитектуры: **3 → 8 → 1**. В скрытом слое оставьте сигмоиду.
3. Для линейного выхода ответ — взвешенная сумма без сигмоиды, а производная активации равна 1.
   При обучении выходная поправка равна `target - output`. Вместо `accuracy` оценивайте среднюю
   абсолютную ошибку **MAE**: среднее значение `abs(правильный ответ - прогноз)` на тестовых данных.
4. Сохраните и загрузите параметры так же, как для классификации.
5. Сделайте HTML-форму. После отправки показывайте **одно предсказанное число** и единицу измерения.
   Не используйте `argmax`: ответ регрессионной сети — значение её единственного выходного нейрона.

### Интерфейс через шаблоны

Признаки вводятся в браузере, а результат отображается на HTML-странице.
Используйте обычные формы `<form method="post">` и рендеринг шаблонов на сервере.
Отдельный клиент и JavaScript для отправки запросов не требуются.

В FastAPI рендеринг шаблона выполняется через **`Jinja2Templates` и `TemplateResponse`**.
Это аналог подхода с `render_template`. Поля формы обработчик принимает через **`Form`**.
[Шаблоны FastAPI](https://fastapi.tiangolo.com/advanced/templates/),
[приём данных формы](https://fastapi.tiangolo.com/tutorial/request-forms/).

Например, результат передаётся в шаблон так:

```python
return templates.TemplateResponse(
    request=request,
    name="index.html",
    context={"prediction": prediction},
)
```

А в HTML-шаблоне выводится так:

```html
<p>Результат: {{ prediction }}</p>
```

Для приложения понадобятся пакеты:

```bash
pip install fastapi uvicorn jinja2 python-multipart numpy
```

**Обучение и сохранение выполняются в Colab. Приложение FastAPI запускается на компьютере.**
Перенесите в приложение классы нейросетей и код загрузки, а скачанные JSON-файлы положите рядом.
При обработке формы используйте введённые значения и сохранённую нормализацию.
Повторное обучение при отправке формы не выполняется.

**Что сдавать:** ноутбук с обучением обеих сетей, код FastAPI, HTML-шаблоны, JSON-файлы
и виде скринкаст (Покажите работу обеих форм минимум на трёх примерах каждой.)


## 9. Решение: регрессия 3 → 8 → 1
Предсказываем длину лепестка (см). Вход: длина чашелистика, ширина чашелистика и ширина лепестка; целевой признак исключён. Используем то же разбиение 120/30, минимумы и максимумы считаем только по обучению. Цель оставляем в сантиметрах. Скрытый слой — сигмоида, выход — линейный. Оптимизируем половину квадрата ошибки, оцениваем MAE.


In [12]:


class RegressionNetwork(SimpleNeuralNetwork):
    """Сигмоидный скрытый слой, ровно один линейный выход."""
    def __init__(self, input_size=3, hidden_size=8):
        super().__init__(input_size, hidden_size, 1)

    def feedforward(self, inputs):
        hidden = np.array([n.feedforward(inputs) for n in self.hidden_layer])
        neuron = self.output_layer[0]
        return np.array([np.dot(hidden, neuron.weights) + neuron.bias])

    def predict(self, inputs):
        return float(self.feedforward(inputs)[0])

    def train(self, inputs, targets, learning_rate=0.02, epochs=1000):
        for epoch in range(epochs):
            for x, target in zip(inputs, targets):
                hidden = np.array([n.feedforward(x) for n in self.hidden_layer])
                out = self.output_layer[0]
                prediction = np.dot(hidden, out.weights) + out.bias
                # Производная линейной активации равна 1.
                delta = float(target) - prediction
                delta_hidden = delta * out.weights.copy() * sigmoid_derivative(hidden)
                out.weights += learning_rate * delta * hidden
                out.bias += learning_rate * delta
                for k, neuron in enumerate(self.hidden_layer):
                    neuron.weights += learning_rate * delta_hidden[k] * x
                    neuron.bias += learning_rate * delta_hidden[k]
            if epoch % 200 == 0 or epoch == epochs - 1:
                print(f"эпоха {epoch:4d} | MAE на обучении {self.mae(inputs, targets):.4f} см")

    def mae(self, inputs, targets):
        return float(np.mean(np.abs(np.array([self.predict(x) for x in inputs]) - targets)))


def save_regression(net, mn, mx, path):
    payload = {
        "hidden": [{"weights": n.weights.tolist(), "bias": float(n.bias)} for n in net.hidden_layer],
        "output": [{"weights": n.weights.tolist(), "bias": float(n.bias)} for n in net.output_layer],
        "mn": mn.tolist(), "mx": mx.tolist(),
        "features": ["sepal_length", "sepal_width", "petal_width"],
        "target": "petal_length", "units": "cm", "output_activation": "linear"
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_model(path, regression=False):
    with open(path, encoding="utf-8") as f:
        saved = json.load(f)
    net = RegressionNetwork() if regression else SimpleNeuralNetwork(4, 8, 3)
    for layer, key in [(net.hidden_layer, "hidden"), (net.output_layer, "output")]:
        if len(layer) != len(saved[key]):
            raise ValueError("Неверная архитектура модели")
        for neuron, params in zip(layer, saved[key]):
            weights = np.asarray(params["weights"], dtype=float)
            if weights.shape != neuron.weights.shape or not np.isfinite(weights).all():
                raise ValueError("Неверные веса модели")
            neuron.weights = weights
            neuron.bias = float(params["bias"])
    mn, mx = np.array(saved["mn"]), np.array(saved["mx"])
    if mn.shape != (3 if regression else 4,) or mx.shape != mn.shape or not np.all(mx > mn):
        raise ValueError("Неверные параметры нормализации")
    return net, mn, mx, saved


In [13]:
# X уже перемешан в разделе 1, но остаётся в исходных единицах.
reg_X = X[:, [0, 1, 3]]
reg_y = X[:, 2]
reg_mn, reg_mx = reg_X[:120].min(axis=0), reg_X[:120].max(axis=0)
reg_train = (reg_X[:120] - reg_mn) / (reg_mx - reg_mn)
reg_test = (reg_X[120:] - reg_mn) / (reg_mx - reg_mn)
np.random.seed(7)
reg_net = RegressionNetwork(3, 8)
reg_net.train(reg_train, reg_y[:120], learning_rate=0.02, epochs=1000)
reg_mae = reg_net.mae(reg_test, reg_y[120:])
baseline_mae = np.mean(np.abs(reg_y[120:] - reg_y[:120].mean()))
print(f"MAE на тесте: {reg_mae:.4f} см")
print(f"MAE константного прогноза (среднее обучения): {baseline_mae:.4f} см")
save_regression(reg_net, reg_mn, reg_mx, "regression_weights.json")
loaded_reg, loaded_mn, loaded_mx, _ = load_model("regression_weights.json", regression=True)
pred_before = np.array([reg_net.predict(x) for x in reg_test])
pred_after = np.array([loaded_reg.predict((x-loaded_mn)/(loaded_mx-loaded_mn)) for x in reg_X[120:]])
np.testing.assert_allclose(pred_before, pred_after, rtol=0, atol=0)
assert len(loaded_reg.output_layer) == 1
print("Проверка восстановления регрессионной сети пройдена.")


эпоха    0 | MAE на обучении 1.3521 см
эпоха  200 | MAE на обучении 0.2330 см
эпоха  400 | MAE на обучении 0.2250 см
эпоха  600 | MAE на обучении 0.2160 см
эпоха  800 | MAE на обучении 0.2077 см
эпоха  999 | MAE на обучении 0.2010 см
MAE на тесте: 0.2223 см
MAE константного прогноза (среднее обучения): 1.7577 см
Проверка восстановления регрессионной сети пройдена.


## 10. Три примера для каждой задачи

In [14]:
examples = [[5.1, 3.5, 1.4, 0.2], [6.0, 2.9, 4.5, 1.5], [6.7, 3.3, 5.7, 2.1]]
for flower in examples:
    x = np.array(flower)
    label = new_names[new_net.predict((x-new_mn)/(new_mx-new_mn))]
    value = loaded_reg.predict((x[[0,1,3]]-loaded_mn)/(loaded_mx-loaded_mn))
    print(f"{flower}: класс {label}; длина лепестка {value:.3f} см (факт {x[2]} см)")


[5.1, 3.5, 1.4, 0.2]: класс setosa; длина лепестка 1.404 см (факт 1.4 см)
[6.0, 2.9, 4.5, 1.5]: класс versicolor; длина лепестка 4.523 см (факт 4.5 см)
[6.7, 3.3, 5.7, 2.1]: класс virginica; длина лепестка 5.506 см (факт 5.7 см)


## 11. Готовое приложение FastAPI
Ячейка создаёт код, шаблон и архив с весами. На компьютере: `python -m pip install -r requirements.txt`, затем `python -m uvicorn main:app --reload`. Откройте http://127.0.0.1:8000. Модели загружаются один раз при запуске; формы используют POST и серверный шаблон Jinja2.

Документация: https://fastapi.tiangolo.com/advanced/templates/ и https://fastapi.tiangolo.com/tutorial/request-forms/.

In [15]:
from pathlib import Path
import zipfile

p = Path('models.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('import json\nimport numpy as np\n\n# Функция активации (сигмоида)\ndef sigmoid(x):\n    return 1 / (1 + np.exp(-x))\n\n# Производная сигмоиды.\n# Принимает УЖЕ посчитанный ответ нейрона s = sigmoid(x): производная равна s * (1 - s)\ndef sigmoid_derivative(s):\n    return s * (1 - s)\n\n\n# Класс нейрона — такой же, как в лабе 1\nclass Neuron:\n    def __init__(self, input_size):\n        # Случайные веса от -1 до 1: и положительные, и отрицательные,\n        # чтобы нейроны с самого начала «смотрели» на признаки по-разному\n        self.weights = np.random.uniform(-1, 1, input_size)\n        self.bias = np.random.uniform(-1, 1)\n\n    def feedforward(self, inputs):\n        total = np.dot(inputs, self.weights) + self.bias   # взвешенная сумма\n        return sigmoid(total)                              # ответ от 0 до 1\n\n# Класс простой нейронной сети с одним скрытым слоем\nclass SimpleNeuralNetwork:\n    def __init__(self, input_size, hidden_size, output_size):\n        # Скрытый слой: hidden_size нейронов, каждый смотрит на все входы\n        self.hidden_layer = [Neuron(input_size) for _ in range(hidden_size)]\n        # Выходной слой: output_size нейронов, каждый смотрит на все скрытые нейроны\n        self.output_layer = [Neuron(hidden_size) for _ in range(output_size)]\n\n    def feedforward(self, inputs):\n        # Прямой проход через скрытый слой\n        hidden_outputs = np.array([neuron.feedforward(inputs) for neuron in self.hidden_layer])\n        # Прямой проход через выходной слой\n        outputs = np.array([neuron.feedforward(hidden_outputs) for neuron in self.output_layer])\n        return outputs\n\n    def predict(self, inputs):\n        # Номер вида, чей выходной нейрон выдал самое большое число\n        return int(np.argmax(self.feedforward(inputs)))\n\n    def accuracy(self, X, labels):\n        # Доля цветков, для которых сеть назвала правильный вид\n        predictions = np.array([self.predict(x) for x in X])\n        return np.mean(predictions == labels)\n\n    def train(self, inputs, targets, learning_rate, epochs):\n        for epoch in range(epochs):\n            total_error = 0\n            for i in range(len(inputs)):\n                # 1. Прямой проход (запоминаем ответы скрытого слоя — они нужны для обучения)\n                hidden_outputs = np.array([neuron.feedforward(inputs[i]) for neuron in self.hidden_layer])\n                outputs = np.array([neuron.feedforward(hidden_outputs) for neuron in self.output_layer])\n\n                # 2. Ошибка на выходном слое: правильный ответ минус ответ сети\n                errors = targets[i] - outputs\n                total_error += np.mean(errors ** 2)\n\n                # 3. Поправка для выходных нейронов: ошибка × производная сигмоиды\n                d_outputs = errors * sigmoid_derivative(outputs)\n\n                # 4. Ошибка течёт назад: каждый скрытый нейрон k собирает вину\n                #    всех выходных нейронов j пропорционально весу связи между ними.\n                #    Считаем ДО изменения весов выходного слоя!\n                d_hidden = np.array([\n                    sum(d_outputs[j] * self.output_layer[j].weights[k] for j in range(len(outputs)))\n                    * sigmoid_derivative(hidden_outputs[k])\n                    for k in range(len(self.hidden_layer))\n                ])\n\n                # 5. Подправляем веса выходного слоя (вход для них — ответы скрытого слоя)\n                for j, neuron in enumerate(self.output_layer):\n                    neuron.weights += learning_rate * d_outputs[j] * hidden_outputs\n                    neuron.bias += learning_rate * d_outputs[j]\n\n                #    ...и веса скрытого слоя (вход для них — признаки цветка)\n                for k, neuron in enumerate(self.hidden_layer):\n                    neuron.weights += learning_rate * d_hidden[k] * inputs[i]\n                    neuron.bias += learning_rate * d_hidden[k]\n\n            # Логирование процесса обучения\n            if epoch % 100 == 0 or epoch == epochs - 1:\n                mse = total_error / len(inputs)\n                acc = self.accuracy(inputs, targets.argmax(axis=1))\n                print(f"эпоха {epoch:4d} | ошибка {mse:.4f} | точность на обучении {acc:.0%}")\n\n\nclass RegressionNetwork(SimpleNeuralNetwork):\n    """Сигмоидный скрытый слой, ровно один линейный выход."""\n    def __init__(self, input_size=3, hidden_size=8):\n        super().__init__(input_size, hidden_size, 1)\n\n    def feedforward(self, inputs):\n        hidden = np.array([n.feedforward(inputs) for n in self.hidden_layer])\n        neuron = self.output_layer[0]\n        return np.array([np.dot(hidden, neuron.weights) + neuron.bias])\n\n    def predict(self, inputs):\n        return float(self.feedforward(inputs)[0])\n\n    def train(self, inputs, targets, learning_rate=0.02, epochs=1000):\n        for epoch in range(epochs):\n            for x, target in zip(inputs, targets):\n                hidden = np.array([n.feedforward(x) for n in self.hidden_layer])\n                out = self.output_layer[0]\n                prediction = np.dot(hidden, out.weights) + out.bias\n                # Производная линейной активации равна 1.\n                delta = float(target) - prediction\n                delta_hidden = delta * out.weights.copy() * sigmoid_derivative(hidden)\n                out.weights += learning_rate * delta * hidden\n                out.bias += learning_rate * delta\n                for k, neuron in enumerate(self.hidden_layer):\n                    neuron.weights += learning_rate * delta_hidden[k] * x\n                    neuron.bias += learning_rate * delta_hidden[k]\n            if epoch % 200 == 0 or epoch == epochs - 1:\n                print(f"эпоха {epoch:4d} | MAE на обучении {self.mae(inputs, targets):.4f} см")\n\n    def mae(self, inputs, targets):\n        return float(np.mean(np.abs(np.array([self.predict(x) for x in inputs]) - targets)))\n\n\ndef save_regression(net, mn, mx, path):\n    payload = {\n        "hidden": [{"weights": n.weights.tolist(), "bias": float(n.bias)} for n in net.hidden_layer],\n        "output": [{"weights": n.weights.tolist(), "bias": float(n.bias)} for n in net.output_layer],\n        "mn": mn.tolist(), "mx": mx.tolist(),\n        "features": ["sepal_length", "sepal_width", "petal_width"],\n        "target": "petal_length", "units": "cm", "output_activation": "linear"\n    }\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(payload, f, ensure_ascii=False, indent=2)\n\n\ndef load_model(path, regression=False):\n    with open(path, encoding="utf-8") as f:\n        saved = json.load(f)\n    net = RegressionNetwork() if regression else SimpleNeuralNetwork(4, 8, 3)\n    for layer, key in [(net.hidden_layer, "hidden"), (net.output_layer, "output")]:\n        if len(layer) != len(saved[key]):\n            raise ValueError("Неверная архитектура модели")\n        for neuron, params in zip(layer, saved[key]):\n            weights = np.asarray(params["weights"], dtype=float)\n            if weights.shape != neuron.weights.shape or not np.isfinite(weights).all():\n                raise ValueError("Неверные веса модели")\n            neuron.weights = weights\n            neuron.bias = float(params["bias"])\n    mn, mx = np.array(saved["mn"]), np.array(saved["mx"])\n    if mn.shape != (3 if regression else 4,) or mx.shape != mn.shape or not np.all(mx > mn):\n        raise ValueError("Неверные параметры нормализации")\n    return net, mn, mx, saved\n', encoding="utf-8")
p = Path('main.py')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text("from contextlib import asynccontextmanager\nfrom pathlib import Path\nfrom typing import Annotated\nimport numpy as np\nfrom fastapi import FastAPI, Request, Form\nfrom fastapi.exceptions import RequestValidationError\nfrom fastapi.responses import HTMLResponse, RedirectResponse\nfrom fastapi.templating import Jinja2Templates\nfrom models import load_model\n\nBASE = Path(__file__).resolve().parent\ntemplates = Jinja2Templates(directory=str(BASE / 'templates'))\nFIELDS = [('sepal_length', 'Длина чашелистика'), ('sepal_width', 'Ширина чашелистика'),\n          ('petal_length', 'Длина лепестка'), ('petal_width', 'Ширина лепестка')]\n\n@asynccontextmanager\nasync def lifespan(app):\n    # Загрузка один раз при старте. Обучения в приложении нет.\n    app.state.classifier = load_model(BASE / 'iris_weights.json')\n    app.state.regressor = load_model(BASE / 'regression_weights.json', regression=True)\n    yield\n\napp = FastAPI(title='Ирисы: классификация и регрессия', lifespan=lifespan)\nMeasurement = Annotated[float, Form(gt=0, le=30, allow_inf_nan=False)]\n\ndef render(request, regression=False, values=None, prediction=None, error=None, status=200):\n    return templates.TemplateResponse(request=request, name='form.html', status_code=status, context={\n        'regression': regression,\n        'fields': [f for f in FIELDS if not regression or f[0] != 'petal_length'],\n        'values': values or {}, 'prediction': prediction, 'error': error})\n\n@app.exception_handler(RequestValidationError)\nasync def validation_error(request, exc):\n    return render(request, regression=request.url.path == '/regression',\n                  error='Введите все размеры числами больше 0 и не больше 30 см.', status=422)\n\n@app.get('/')\ndef index():\n    return RedirectResponse('/classification')\n\n@app.get('/classification', response_class=HTMLResponse)\ndef classification_page(request: Request):\n    return render(request)\n\n@app.post('/classification', response_class=HTMLResponse)\ndef classification(request: Request, sepal_length: Measurement, sepal_width: Measurement,\n                   petal_length: Measurement, petal_width: Measurement):\n    values = dict(zip([f[0] for f in FIELDS], [sepal_length, sepal_width, petal_length, petal_width]))\n    net, mn, mx, metadata = request.app.state.classifier\n    x = np.array(list(values.values()))\n    prediction = metadata['names'][net.predict((x - mn) / (mx - mn))]\n    return render(request, values=values, prediction=prediction)\n\n@app.get('/regression', response_class=HTMLResponse)\ndef regression_page(request: Request):\n    return render(request, regression=True)\n\n@app.post('/regression', response_class=HTMLResponse)\ndef regression(request: Request, sepal_length: Measurement, sepal_width: Measurement, petal_width: Measurement):\n    values = dict(sepal_length=sepal_length, sepal_width=sepal_width, petal_width=petal_width)\n    net, mn, mx, _ = request.app.state.regressor\n    value = net.predict((np.array(list(values.values())) - mn) / (mx - mn))\n    return render(request, regression=True, values=values, prediction=f'{value:.3f} см')\n", encoding="utf-8")
p = Path('templates/form.html')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('<!doctype html>\n<html lang="ru">\n<head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<title>{{ \'Регрессия\' if regression else \'Классификация\' }} ирисов</title>\n<style>\n*{box-sizing:border-box}body{margin:0;background:#f0f4f7;color:#172d3a;font:17px/1.5 system-ui,sans-serif}main{max-width:760px;margin:40px auto;padding:32px;background:white;border-radius:20px;box-shadow:0 12px 50px #172d3a0d}nav{display:flex;gap:12px}nav a{color:#176f69;text-decoration:none;padding:10px 16px;border:1px solid #bcd4d0;border-radius:10px}nav a[aria-current=page]{background:#176f69;color:white}h1{font-size:30px;line-height:1.2;margin-bottom:12px}.eyebrow{font-size:12px;letter-spacing:2px;color:#617681;margin:26px 0 12px}.intro{color:#617681}form{display:grid;grid-template-columns:1fr 1fr;gap:18px}label{display:block;font-size:15px}input{display:block;width:100%;padding:12px;margin-top:6px;border:1px solid #c9d5dd;border-radius:8px;font:inherit}button{grid-column:1/-1;background:#176f69;color:white;border:0;border-radius:9px;padding:14px;font:inherit;cursor:pointer}.result{padding:18px 22px;background:#e8f5ee;border-left:4px solid #176f69;border-radius:8px;margin-top:24px}.result strong{font-size:25px;display:block}.error{color:#9a2733;background:#fff1f1;padding:14px}.note{font-size:13px;color:#617681;margin-top:24px}@media(max-width:600px){main{margin:15px;padding:20px}form{grid-template-columns:1fr}nav{flex-wrap:wrap}}\n</style></head>\n<body><main><nav><a href="/classification" {% if not regression %}aria-current="page"{% endif %}>Классификация</a><a href="/regression" {% if regression %}aria-current="page"{% endif %}>Регрессия</a></nav>\n<p class="eyebrow">ЛАБОРАТОРНАЯ РАБОТА № 4 · IRIS</p>\n<h1>{{ \'Предсказать длину лепестка\' if regression else \'Определить вид ириса\' }}</h1>\n<p class="intro">{{ \'Введите три измерения. Модель предскажет длину лепестка в сантиметрах.\' if regression else \'Введите четыре измерения цветка. Модель определит один из трёх видов ириса.\' }}</p>\n{% if error %}<p role="alert" class="error">{{ error }}</p>{% endif %}\n<form method="post" action="{{ \'/regression\' if regression else \'/classification\' }}">\n{% for name, label in fields %}<label for="{{ name }}">{{ label }}, см<input id="{{ name }}" name="{{ name }}" type="number" step="any" min="0.001" max="30" required value="{{ values.get(name, \'\') }}"></label>{% endfor %}\n<button type="submit">{{ \'Рассчитать длину\' if regression else \'Определить вид\' }}</button>\n</form>\n{% if prediction is not none %}<div class="result" role="status">Результат<strong id="prediction">{{ prediction }}</strong></div>{% endif %}\n<p class="note">Учебная модель на наборе Iris. Для измерений, сильно отличающихся от обучающих данных, прогноз может быть неточным.</p>\n</main></body></html>\n', encoding="utf-8")
p = Path('requirements.txt')
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('numpy>=1.24,<3\nfastapi>=0.108,<1\nuvicorn>=0.23,<1\njinja2>=3.1,<4\npython-multipart>=0.0.9,<1\n', encoding="utf-8")
with zipfile.ZipFile("fastapi_app.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["models.py", "main.py", "templates/form.html", "requirements.txt", "iris_weights.json", "regression_weights.json"]:
        archive.write(name)
print("Созданы исходники приложения и fastapi_app.zip")


Созданы исходники приложения и fastapi_app.zip


In [16]:
try:
    from google.colab import files
except ImportError:
    print("Скачайте fastapi_app.zip из текущей папки.")
else:
    files.download("fastapi_app.zip")


Скачайте fastapi_app.zip из текущей папки.


## 12. Вывод
Обучены и сохранены две сети. Классификация: 29/30 правильных ответов (96,67%). Регрессия: MAE 0,2223 см против 1,7577 см у константного прогноза. Регрессия имеет один линейный выход; целевой признак исключён из входов. После загрузки JSON предсказания совпадают на всех тестовых объектах. Реализованы две HTML-формы FastAPI с серверным рендерингом. Для сдачи необходимо дополнительно записать скринкаст: сценарий и шесть примеров находятся в SCREENCAST.md в полном архиве.